# Negocio: estrategias centrales y robustez
Ejecutar después de `03_modelo.ipynb`. Lee Gold y clusters locales, valida los 785 municipios y guarda resultados en `data/outputs/`, sin S3.

El escenario principal usa los supuestos centrales y compara el baseline obligatorio, máxima facturación sin clusters, Modelo A oficial y la alternativa diversificada por clusters. Modelo A mantiene el filtro oficial por media del cluster; la diversificación se presenta como una opción de cartera, no como sustitución.

La sensibilidad queda separada como validación. El informe final está en `data/outputs/resumen_negocio.md`.

In [10]:
import importlib
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next((path for path in (working_directory, *working_directory.parents)
                     if (path / 'src' / 'negocio' / 'ejecutar.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Could not find project root containing src/negocio.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.negocio import ejecutar, valor_negocio
importlib.reload(valor_negocio)
ejecutar = importlib.reload(ejecutar)
SALIDA = ejecutar.SALIDA
main = ejecutar.main


In [11]:
resultados_negocio = main()

Clusters leídos de clusters.csv: 4 grupos
Entrada: C:\Users\d.gonzalez.fuentes\OneDrive - Accenture\Documents\market-expansion-sot-modelo-negocio-baseline\data\processed\gold\tabla_oro_municipios_andalucia.parquet · 785 municipios · clusters: sí
Salidas T5 anteriores reemplazadas: baseline_central.csv, comparacion_estrategias_central.csv, municipios_evaluados_central.csv, resumen_negocio.md, seleccion_diversificada_clusters_central.csv, seleccion_modelo_central.csv, sensibilidad_detalle.csv, sensibilidad_frecuencias.csv, top_facturacion_central.csv
Comparación de estrategias centrales:
                            estrategia  centros_seleccionados  centros_viables  facturacion_anual_total_con_tope_eur  margen_operativo_anual_total_eur  perdidas_operativas_totales_eur  margen_medio_por_centro_eur  demanda_media_por_competidor                                  municipios_con_aviso_osm  clusters_representados cluster_sin_candidato_viable
                              Baseline               

In [13]:
comparacion_view = resultados_negocio['comparacion'].drop(
    columns=['cluster_sin_candidato_viable']
 ).rename(columns={
    'estrategia': 'Estrategia',
    'centros_seleccionados': 'Centros seleccionados',
    'centros_viables': 'Centros viables',
    'facturacion_anual_total_con_tope_eur': 'Facturación anual total con tope (EUR)',
    'margen_operativo_anual_total_eur': 'Margen operativo anual total (EUR)',
    'perdidas_operativas_totales_eur': 'Pérdidas operativas totales (EUR)',
    'margen_medio_por_centro_eur': 'Margen medio por centro (EUR)',
    'demanda_media_por_competidor': 'Demanda media por competidor',
    'municipios_con_aviso_osm': 'Municipios con aviso OSM',
    'clusters_representados': 'Clusters representados',
})
for columna in [
    'Facturación anual total con tope (EUR)', 'Margen operativo anual total (EUR)',
    'Pérdidas operativas totales (EUR)', 'Margen medio por centro (EUR)',
]:
    comparacion_view[columna] = comparacion_view[columna].map(
        lambda valor: ejecutar.formato_con_sufijo(valor, 0, ' EUR')
    )
comparacion_view['Demanda media por competidor'] = comparacion_view[
    'Demanda media por competidor'
].map(lambda valor: ejecutar.formato_numero(valor, 1))
display(Markdown('## Comparación de estrategias'))
display(comparacion_view)

columnas_seleccion = [
    'municipality_name', 'cluster', 'population_18_50', 'competidores_pond',
    'socios_captables', 'facturacion_sin_tope_capacidad', 'ratio_capacidad',
    'comp_per_10k_18_50', 'media_cluster_comp_per_10k', 'colchon_demanda_pct',
    'facturacion_anual', 'margen_eur', 'aviso_calidad',
]
nombres_columnas = {
    'municipality_name': 'Municipio',
    'cluster': 'Cluster',
    'population_18_50': 'Población 18-50',
    'competidores_pond': 'Competidores ponderados',
    'socios_captables': 'Socios captables sin tope',
    'facturacion_sin_tope_capacidad': 'Potencial teórico sin límite de capacidad (EUR)',
    'ratio_capacidad': 'Ratio de capacidad',
    'comp_per_10k_18_50': 'Competencia por 10.000',
    'media_cluster_comp_per_10k': 'Media de competencia del cluster',
    'colchon_demanda_pct': 'Colchón de demanda',
    'facturacion_anual': 'Facturación anual con tope (EUR)',
    'margen_eur': 'Margen anual (EUR)',
    'aviso_calidad': 'Aviso OSM',
}
def formatear_seleccion(vista):
    vista = vista.reset_index(drop=True).copy()
    for columna in [
        'Población 18-50', 'Socios captables sin tope',
        'Potencial teórico sin límite de capacidad (EUR)',
        'Facturación anual con tope (EUR)', 'Margen anual (EUR)',
    ]:
        vista[columna] = vista[columna].map(ejecutar.formato_numero)
    vista['Competidores ponderados'] = vista['Competidores ponderados'].map(
        lambda valor: ejecutar.formato_numero(valor, 1)
    )
    vista['Ratio de capacidad'] = vista['Ratio de capacidad'].map(
        lambda valor: ejecutar.formato_con_sufijo(valor, 2, '×')
    )
    vista['Competencia por 10.000'] = vista['Competencia por 10.000'].map(
        lambda valor: ejecutar.formato_numero(valor, 2)
    )
    vista['Media de competencia del cluster'] = vista[
        'Media de competencia del cluster'
    ].map(lambda valor: ejecutar.formato_numero(valor, 2))
    vista['Colchón de demanda'] = vista['Colchón de demanda'].map(
        lambda valor: ejecutar.formato_con_sufijo(valor * 100, 1, '%')
    )
    vista['Aviso OSM'] = vista['Aviso OSM'].fillna('')
    return vista

modelo_a_view = resultados_negocio['selecciones']['modelo_a'][columnas_seleccion].rename(
    columns=nombres_columnas
 )
modelo_a_view.insert(0, 'Posición', range(1, len(modelo_a_view) + 1))
modelo_a_view = formatear_seleccion(modelo_a_view)
display(Markdown('## Modelo A oficial'))
display(modelo_a_view)

columnas_diversificada = ['posicion', *columnas_seleccion, 'motivo_seleccion']
diversificada_view = resultados_negocio['selecciones']['diversificada'][
    columnas_diversificada
].rename(columns={'posicion': 'Posición', **nombres_columnas,
                'motivo_seleccion': 'Motivo de selección'})
diversificada_view = formatear_seleccion(diversificada_view)
display(Markdown('## Alternativa diversificada por clusters'))
display(diversificada_view)

frecuencias_view = resultados_negocio['sensibilidad_frecuencias'][[
    'municipality_name', 'apariciones', 'posicion_media', 'fact_min', 'fact_max',
    'robustez', 'aviso_calidad',
]].head(10).rename(columns={
    'municipality_name': 'Municipio',
    'apariciones': 'Apariciones',
    'posicion_media': 'Posición media',
    'fact_min': 'Facturación mínima (EUR)',
    'fact_max': 'Facturación máxima (EUR)',
    'robustez': 'Robustez',
    'aviso_calidad': 'Aviso OSM',
}).reset_index(drop=True)
frecuencias_view['Posición media'] = frecuencias_view['Posición media'].map(
    lambda valor: ejecutar.formato_numero(valor, 1)
 )
for columna in ['Facturación mínima (EUR)', 'Facturación máxima (EUR)']:
    frecuencias_view[columna] = frecuencias_view[columna].map(
        ejecutar.formato_numero
    )
frecuencias_view['Apariciones'] = frecuencias_view['Apariciones'].map(
    ejecutar.formato_numero
 )
frecuencias_view['Aviso OSM'] = frecuencias_view['Aviso OSM'].fillna('')
display(Markdown('## Robustez: diez municipios principales'))
display(frecuencias_view)

report_path = resultados_negocio['resumen_path']
display(Markdown(f'Informe completo: `{report_path}`'))

## Comparación de estrategias

,Estrategia,Centros seleccionados,Centros viables,Facturación anual total con tope (EUR),Margen operativo anual total (EUR),Pérdidas operativas totales (EUR),Margen medio por centro (EUR),Demanda media por competidor,Municipios con aviso OSM,Clusters representados
0,Baseline,5,3,2.649.735 EUR,-130.265 EUR,330.967 EUR,-26.053 EUR,"973,8",,1
1,Máxima facturación sin clusters,5,5,3.489.720 EUR,709.720 EUR,0 EUR,141.944 EUR,"4.538,8","Alcalá de Guadaíra, Fuengirola, Puerto de Sant...",1
2,Modelo A oficial,5,5,3.489.720 EUR,709.720 EUR,0 EUR,141.944 EUR,"4.538,8","Alcalá de Guadaíra, Fuengirola, Puerto de Sant...",1
3,Alternativa diversificada por clusters,5,5,3.489.720 EUR,709.720 EUR,0 EUR,141.944 EUR,"3.529,4","Fuengirola, Puerto de Santa María, El",2


## Modelo A oficial

,Posición,Municipio,Cluster,Población 18-50,Competidores ponderados,Socios captables sin tope,Potencial teórico sin límite de capacidad (EUR),Ratio de capacidad,Competencia por 10.000,Media de competencia del cluster,Colchón de demanda,Facturación anual con tope (EUR),Margen anual (EUR),Aviso OSM
0,1,Marbella,2,72.448,"5,9",4.261,1.143.866,"1,64×","0,81","2,40","51,4%",697.944,141.944,
1,2,"Puerto de Santa María, El",2,38.048,"1,2",3.318,890.626,"1,28×","0,32","2,40","37,6%",697.944,141.944,Sin gimnasios en OSM: validar con una segunda ...
2,3,Algeciras,2,55.729,"6,2",3.160,848.394,"1,22×","1,11","2,40","34,5%",697.944,141.944,
3,4,Fuengirola,2,34.640,"2,1",3.021,810.852,"1,16×","0,61","2,40","31,4%",697.944,141.944,Sin gimnasios en OSM: validar con una segunda ...
4,5,Alcalá de Guadaíra,2,34.632,"0,7",3.020,810.665,"1,16×","0,20","2,40","31,4%",697.944,141.944,Sin gimnasios en OSM: validar con una segunda ...


## Alternativa diversificada por clusters

,Posición,Municipio,Cluster,Población 18-50,Competidores ponderados,Socios captables sin tope,Potencial teórico sin límite de capacidad (EUR),Ratio de capacidad,Competencia por 10.000,Media de competencia del cluster,Colchón de demanda,Facturación anual con tope (EUR),Margen anual (EUR),Aviso OSM,Motivo de selección
0,1,Sanlúcar de Barrameda,1,29.848,"2,6",2.603,698.681,"1,00×","0,87","3,18","20,4%",697.944,141.944,,mejor_candidato_viable_del_cluster
1,2,Marbella,2,72.448,"5,9",4.261,1.143.866,"1,64×","0,81","2,40","51,4%",697.944,141.944,,mejor_candidato_viable_del_cluster
2,3,"Puerto de Santa María, El",2,38.048,"1,2",3.318,890.626,"1,28×","0,32","2,40","37,6%",697.944,141.944,Sin gimnasios en OSM: validar con una segunda ...,relleno_por_cluster_sin_candidato_viable
3,4,Algeciras,2,55.729,"6,2",3.160,848.394,"1,22×","1,11","2,40","34,5%",697.944,141.944,,relleno_por_cluster_sin_candidato_viable
4,5,Fuengirola,2,34.640,"2,1",3.021,810.852,"1,16×","0,61","2,40","31,4%",697.944,141.944,Sin gimnasios en OSM: validar con una segunda ...,quinta_plaza_mejor_restante


## Robustez: diez municipios principales

,Municipio,Apariciones,Posición media,Facturación mínima (EUR),Facturación máxima (EUR),Robustez,Aviso OSM
0,Marbella,100,"1,7",614.442,697.944,100 de 162,
1,Algeciras,59,"3,1",617.096,697.944,59 de 162,
2,"Puerto de Santa María, El",58,"2,6",556.641,697.944,58 de 162,Sin gimnasios en OSM: validar con una segunda ...
3,Alcalá de Guadaíra,52,"2,8",599.165,697.944,52 de 162,Sin gimnasios en OSM: validar con una segunda ...
4,Huelva,44,"3,1",592.270,697.944,44 de 162,
5,Fuengirola,44,"3,9",613.718,697.944,44 de 162,Sin gimnasios en OSM: validar con una segunda ...
6,Dos Hermanas,44,"4,1",564.350,697.944,44 de 162,
7,Jerez de la Frontera,33,"3,0",662.894,697.944,33 de 162,
8,Almería,31,"3,6",654.989,697.944,31 de 162,
9,Córdoba,24,"1,9",697.944,697.944,24 de 162,


Informe completo: `C:\Users\d.gonzalez.fuentes\OneDrive - Accenture\Documents\market-expansion-sot-modelo-negocio-baseline\data\outputs\resumen_negocio.md`